### Projectinformatie
**Opleiding:** Toegepaste Wiskunde & Data Science  
**Vak:** Beroepsproject 3.4  
**Begeleider:** R. Nolet  
**Datum:** 20-03-2026  

| Teamleden             | Studentnummer |
|----------------------|---------------|
| Murielle Tichelaar   | 500926485     |
| Nalini Bisessar      | 500874334     |
| Lushan Strack        | 500934278     |
| Dinand Voogt         | 500934202     |

# Deployment: Random Forest Model

In deze notebook wordt het getrainde Random Forest model toegepast op nieuwe, ongeziene klantdata (`churndata_deployment.csv`).

Het model beslist welke klanten tarief 2 aangeboden moeten worden en schrijft dit weg naar `tarief2.csv`. Daarnaast wordt een winstverwachting berekend en uitgeprint.

### 1. Libraries & Model laden

In [6]:
import pandas as pd
import numpy as np
import joblib

# Model inladen
model = joblib.load("random_forest_model.joblib")
print("Model geladen:", type(model).__name__)

Model geladen: Pipeline


### 2. Deployment data inladen

Het bestand `churndata_deployment.csv` wordt ingeladen. Dit zijn nieuwe klanten zonder bekende churn-status.

In [7]:
df = pd.read_csv("churndata_deployment.csv", sep=",", decimal=".")

X = df.copy()

print(f"Aantal klanten: {len(X)}")
X.head()

FileNotFoundError: [Errno 2] No such file or directory: 'churndata_deployment.csv'

### 3. Voorspelling & Beslissing tarief 2

Het model voorspelt de churnkans per klant. Op basis van een gepersonaliseerde threshold, berekend uit de verwachte winst bij tarief 1 versus tarief 2, wordt beslist wie tarief 2 krijgt aangeboden.

In [ ]:
# Churnkans voorspellen
y_prob = model.predict_proba(X)[:, 1]

# Gepersonaliseerde threshold per klant
W1 = 0.1 * X["Frequency of SMS"] + 0.3 * X["Seconds of Use"]
W2 = 0.07 * X["Frequency of SMS"] + 0.2 * X["Seconds of Use"]

thresholds = (W1 - W2) / (W1 - 0.75 * W2)

# Beslissing: tarief 2 aanbieden als churnkans >= threshold
offer_tarief2 = (y_prob >= thresholds)

print(f"Aantal klanten tarief 2: {offer_tarief2.sum()}")
print(f"Totaal aantal klanten:   {len(X)}")
print(f"Percentage tarief 2:     {offer_tarief2.sum()/len(X):.2%}")

### 4. Wegschrijven naar tarief2.csv

De klanten die tarief 2 aangeboden krijgen worden weggeschreven naar `tarief2.csv`.

In [ ]:
df_tarief2 = X[offer_tarief2].copy()
df_tarief2["churn_kans"] = y_prob[offer_tarief2]

df_tarief2.to_csv("tarief2.csv", index=False)
print(f"tarief2.csv opgeslagen met {len(df_tarief2)} klanten.")

### 5. Winstverwachting

De verwachte maandelijkse winst wordt berekend op basis van de modelstrategie.
Per klant wordt de verwachte winst berekend afhankelijk van het aangeboden tarief.

In [ ]:
sms_pm = X["Frequency of SMS"] / 9
min_pm = (X["Seconds of Use"] / 60) / 9

W1_pm = 0.1 * sms_pm + 0.3 * min_pm
W2_pm = 0.07 * sms_pm + 0.2 * min_pm

# Verwachte winst per klant per maand
E_tarief1 = (1 - y_prob) * W1_pm
E_tarief2 = (1 - 0.75 * y_prob) * W2_pm

# Totale verwachte winst
totale_winst = (
    E_tarief2[offer_tarief2].sum() +
    E_tarief1[~offer_tarief2].sum()
)

print(f"Verwachte maandelijkse winst: €{totale_winst:.2f}")